# WorkflowHub edges vs. citation-order edges

Do tools that run one after the other in Galaxy workflows also get cited in that order?

- **Workflows:** `data/biotools/biotools_edges.csv` (`map_galaxy_tools.py`): tool A feeds tool B in `n_workflows` workflows
- **Citations:** `../edges-processed/directed_edges.csv` (`citation-order/` steps 3 to 5): tool A is cited before tool B in `n_articles` articles

Each workflow pair is put in its main workflow direction, `source` → `target`. Then both sources get a direction ratio, from -1 to 1:
- `wf_dir = (wf_fwd - wf_rev) / (wf_fwd + wf_rev)`: 1 = all workflows run `source` first, 0 = as often both ways. It is never negative, since `source` is the main direction.
- `cit_dir`: the same on the article counts. 1 = always cited `source` first, -1 = always `target` first. Empty when the pair is never cited.

`distance = |wf_dir - cit_dir|` says how well the two agree: 0 = the same ratio, 2 = opposite directions (all workflows one way, all articles the other).

In [103]:
import duckdb

MAX_CONFIDENCE = 3   # workflow edges to use: 1 = Codex mapping only, 2 = + name match, 3 = all (+ prefix match)
MIN_ARTICLES = 10    # pairs with fewer articles are left out of the summary

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE TABLE wf AS
    SELECT lower(source) AS source, lower(target) AS target, n_workflows
    FROM 'data/biotools/biotools_edges.csv' WHERE confidence <= {MAX_CONFIDENCE}
""")
con.execute("CREATE OR REPLACE TABLE cit AS SELECT source, target, n_articles FROM '../edges-processed/directed_edges.csv'")
con.execute("CREATE OR REPLACE TABLE cit_tools AS SELECT source AS tool FROM cit UNION SELECT target FROM cit")

# One row per workflow pair in its main direction, with the counts of both sources both ways.
con.execute(f"""
    CREATE OR REPLACE TABLE pairs AS
    WITH counts AS (
        SELECT f.source, f.target,
               f.n_workflows AS wf_fwd, coalesce(r.n_workflows, 0) AS wf_rev,
               coalesce(cf.n_articles, 0) AS cit_fwd, coalesce(cr.n_articles, 0) AS cit_rev,
               f.source IN (SELECT tool FROM cit_tools) AND f.target IN (SELECT tool FROM cit_tools) AS both_in_cit
        FROM wf f
        LEFT JOIN wf r ON r.source = f.target AND r.target = f.source
        LEFT JOIN cit cf ON cf.source = f.source AND cf.target = f.target
        LEFT JOIN cit cr ON cr.source = f.target AND cr.target = f.source
        WHERE f.n_workflows > coalesce(r.n_workflows, 0)                -- the main direction,
           OR (f.n_workflows = r.n_workflows AND f.source < f.target)  -- on a tie the alphabetical one
    ),
    ratios AS (
        SELECT *,
               (wf_fwd - wf_rev) / (wf_fwd + wf_rev) AS wf_dir,
               (cit_fwd - cit_rev) / nullif(cit_fwd + cit_rev, 0) AS cit_dir
        FROM counts
    )
    SELECT *, abs(wf_dir - cit_dir) AS distance FROM ratios
""")

## Per edge

One row per workflow pair, most workflows first.

In [104]:
con.sql("""
    SELECT source, target, wf_fwd, wf_rev, round(wf_dir, 2) AS wf_dir,
           cit_fwd, cit_rev, round(cit_dir, 2) AS cit_dir, round(distance, 2) AS distance
    FROM pairs ORDER BY wf_fwd + wf_rev DESC, cit_fwd + cit_rev DESC
""").show(max_rows=1000)

┌───────────────────────────────┬───────────────────────────────┬────────┬────────┬────────┬─────────┬─────────┬─────────┬──────────┐
│            source             │            target             │ wf_fwd │ wf_rev │ wf_dir │ cit_fwd │ cit_rev │ cit_dir │ distance │
│            varchar            │            varchar            │ int64  │ int64  │ double │  int64  │  int64  │ double  │  double  │
├───────────────────────────────┼───────────────────────────────┼────────┼────────┼────────┼─────────┼─────────┼─────────┼──────────┤
│ anndata                       │ scanpy                        │     24 │     24 │    0.0 │       6 │       2 │     0.5 │      0.5 │
│ fastqc                        │ multiqc                       │     41 │      0 │    1.0 │       0 │       0 │    NULL │     NULL │
│ fastp                         │ multiqc                       │     37 │      0 │    1.0 │      36 │      18 │    0.33 │     0.67 │
│ fastp                         │ bwa                         

## Overlap

How many workflow pairs could be in the citation graph (both tools are in it somewhere), and how many are.

In [105]:
con.sql("""
    UNPIVOT (
        SELECT count(*)                                         AS "workflow pairs",
               count(*) FILTER (both_in_cit)                    AS "both tools in the citation graph",
               count(*) FILTER (cit_dir IS NOT NULL)            AS "also a citation edge",
               count(*) FILTER (wf_dir < 1)                     AS "both ways in workflows"
        FROM pairs
    ) ON COLUMNS(*) INTO NAME measure VALUE pairs
""").show()

┌──────────────────────────────────┬───────┐
│             measure              │ pairs │
│             varchar              │ int64 │
├──────────────────────────────────┼───────┤
│ workflow pairs                   │   712 │
│ both tools in the citation graph │   339 │
│ also a citation edge             │   264 │
│ both ways in workflows           │    46 │
└──────────────────────────────────┴───────┘



## Workflow tools missing from the citation graph

A tool can be missing because it has no publication in bio.tools, or because its papers are never cited in the Methods of the research articles.

In [106]:
con.sql("""
    WITH wf_tools AS (
        SELECT tool, sum(n_workflows) AS n_workflows
        FROM (SELECT source AS tool, n_workflows FROM wf UNION ALL SELECT target, n_workflows FROM wf)
        GROUP BY tool
    ),
    pubs AS (SELECT DISTINCT lower(biotoolsID) AS tool FROM '../biotools/data/biotools_publications.csv')
    SELECT tool IN (SELECT tool FROM cit_tools) AS in_citation_graph,
           tool IN (SELECT tool FROM pubs) AS has_publication,
           count(*) AS tools, sum(n_workflows) AS workflow_edges
    FROM wf_tools GROUP BY ALL ORDER BY ALL
""").show()

┌───────────────────┬─────────────────┬───────┬────────────────┐
│ in_citation_graph │ has_publication │ tools │ workflow_edges │
│      boolean      │     boolean     │ int64 │     int128     │
├───────────────────┼─────────────────┼───────┼────────────────┤
│ false             │ false           │    66 │            435 │
│ false             │ true            │    64 │            583 │
│ true              │ true            │   193 │           2660 │
└───────────────────┴─────────────────┴───────┴────────────────┘



## Summary

For the pairs with at least `MIN_ARTICLES` articles.

In [ ]:
con.sql(f"""
    UNPIVOT (
        SELECT count(*)                                            AS "pairs compared",
               round(avg(distance), 2)                             AS "mean distance (0 to 2 - because "direction" is -1 to 1)",
               round(median(distance), 2)                          AS "median distance",
               FROM pairs WHERE cit_fwd + cit_rev >= {MIN_ARTICLES}
    ) ON COLUMNS(*) INTO NAME measure VALUE value
""").show()

┌────────────────────────┬────────┐
│        measure         │ value  │
│        varchar         │ double │
├────────────────────────┼────────┤
│ pairs compared         │  141.0 │
│ mean distance (0 to 2) │   0.62 │
│ median distance        │   0.45 │
└────────────────────────┴────────┘

